# **Goal**

This notebook applies **clustering** techniques to a dataset of Spotify tracks to identify
**sound profile patterns** and group tracks with similar characteristics. The analysis explores
how audio features relate to each other and lays the groundwork for a **music recommendation
system** based on how songs sound, regardless of genre or popularity.

---

# **About the dataset**

[Spotify Tracks Attributes and Popularity](https://www.kaggle.com/datasets/melissamonfared/spotify-tracks-attributes-and-popularity)
(Kaggle) contains ~114k Spotify tracks across 114 genres, with:

* **Audio features:** danceability, energy, loudness, acousticness, instrumentalness, valence, tempo, among others.
* **Popularity:** Spotify's popularity index.
* **Metadata:** track name, artists, album name, genre.

Spotify no longer exposes audio features to new apps (since November 2024), so this dataset is
the only source of audio features for the project.

---

# **Columns**

1. **track_id:** Spotify track ID (used later to play tracks through the Spotify embed player).
2. **track_name:** Track name.
3. **artists:** Artist(s), separated by `;`.
4. **album_name:** Album the track was released on.
5. **track_genre:** Genre the track was collected under.
6. **popularity:** Spotify popularity index (0 to 100).
7. **duration_ms:** Track length in milliseconds.
8. **explicit:** Whether the track has explicit lyrics.
9. **danceability:** How suitable the track is for dancing (0 to 1).
10. **energy:** Perceived intensity and activity (0 to 1).
11. **key:** Musical key, as a pitch class (0 = C, 1 = C♯/D♭, …, 11 = B).
12. **loudness:** Average loudness (dB).
13. **mode:** Modality (1 = major, 0 = minor).
14. **speechiness:** Presence of spoken words (0 to 1).
15. **acousticness:** Confidence that the track is acoustic (0 to 1).
16. **instrumentalness:** Likelihood that the track has no vocals (0 to 1).
17. **liveness:** Likelihood that the track was recorded live (0 to 1).
18. **valence:** Musical positiveness (0 to 1).
19. **tempo:** Estimated tempo in BPM.
20. **time_signature:** Estimated beats per bar (values 0 to 5 in this dataset).

## **Loading and inspecting the data**

In [ ]:
# Download the dataset from Kaggle (public, no credentials needed)
!pip install -q kagglehub

import kagglehub

# Link: https://www.kaggle.com/datasets/melissamonfared/spotify-tracks-attributes-and-popularity
path = kagglehub.dataset_download('melissamonfared/spotify-tracks-attributes-and-popularity')

import os

# List the downloaded files
for entry in os.listdir(path):
    print(entry)

In [ ]:
import pandas as pd
import os

df = pd.read_csv(os.path.join(path, 'dataset.csv'))

print("First rows of the dataset:")
display(df.head())
print("\nShape:", df.shape)

In [ ]:
print(f"Rows and columns: {df.shape}")

print("\nData types:")
print(df.dtypes)

print("\nMissing values per column:")
print(df.isnull().sum())

print("\nTop 20 genres by number of tracks:")
print(df['track_genre'].value_counts().head(20))

print("\nPopularity distribution:")
print(df['popularity'].describe())

# Pairwise correlations between numeric columns
corr = df.corr(numeric_only=True)
corr_pairs = (
    corr.unstack()
    .sort_values(ascending=False)
    .drop_duplicates()
)
print("\nTop 10 correlations (absolute value) between numeric columns:")
print(corr_pairs[(corr_pairs < 1.0)].abs().sort_values(ascending=False).head(10))

## **Exploratory analysis: first observations**

**1.** Only one row has missing values (in *artists*, *album_name* and *track_name*). These columns are not used as clustering features.

**2.** Every genre appears exactly 1000 times in the raw data. This balance is partly artificial: the same track can be listed under several genres (see *Removing duplicate tracks* below).

**3.** Popularity is spread out and most tracks have low popularity, so the dataset is not made only of hits.

**4. Main correlations:** some audio features are strongly and consistently correlated:
* energy – loudness (0.76): more energetic tracks tend to be louder.
* energy – acousticness (−0.73): acoustic tracks tend to have low energy.
* acousticness – loudness (−0.59): more acoustic tracks tend to be quieter.
* danceability – valence (0.48): more danceable tracks tend to sound "happier".

*Popularity is **not** strongly correlated with any audio feature. A track's success seems to depend on external factors (marketing, artist, playlists, etc.) rather than on how it sounds.*

## **Data cleaning**

In [ ]:
# Drop rows with missing values (only 1 row in this dataset)
df_clean = df.dropna().copy()

print("Rows removed because of missing values:", len(df) - len(df_clean))

### Removing duplicate tracks

The dataset has two kinds of duplicates:

1. **Same `track_id`, different genre.** The dataset was built by collecting ~1000 tracks
   per genre, so a track tagged with several genres appears once per genre
   (e.g. *I'll Be Home For Christmas* by Kacey Musgraves and Lana Del Rey appears 5 times).
   The audio features are identical; only `track_genre` changes.
2. **Same song (name + artists), different `track_id`.** The same recording released as a
   single and on an album gets a different Spotify ID (e.g. *The Hills* by The Weeknd,
   once with popularity 88 and once with popularity 3).

If kept, duplicates would be recommended as "similar" to themselves and would give
multi-genre tracks extra weight in the clustering.

**Decision:** keep one row per track and collect all its genres into a `genres` list
instead of dropping them, since that information is useful later (e.g. genre filters in the API).
When the same song has several IDs, keep the most popular version, which is the one
Spotify surfaces as canonical. `track_genre` keeps the genre of the row that was kept,
so the existing analysis below still works.

In [ ]:
# 1) Same track_id listed under several genres: keep one row, collect all genres.
# Sorting by popularity (stable, with track_id as tie-breaker) makes the kept row deterministic.
rows_before = len(df_clean)

genres_by_id = df_clean.groupby('track_id')['track_genre'].agg(lambda s: sorted(set(s)))

df_clean = (
    df_clean
    .sort_values(['popularity', 'track_id'], ascending=[False, True], kind='stable')
    .drop_duplicates('track_id')
)
df_clean['genres'] = df_clean['track_id'].map(genres_by_id)

print(f"Rows removed (duplicate track_id): {rows_before - len(df_clean):,}")
print(f"Tracks left: {len(df_clean):,}")
print(f"Tracks with more than one genre: {(df_clean['genres'].str.len() > 1).sum():,}")

In [ ]:
# 2) Same song (name + artists) under different track_ids: keep the most popular version,
# merging the genres of all versions.
song_key = ['track_name', 'artists']
rows_before = len(df_clean)

print("Example before:")
display(df_clean.loc[df_clean['track_name'] == 'The Hills', ['track_id', 'artists', 'album_name', 'popularity']])

genres_by_song = (
    df_clean.groupby(song_key)['genres']
    .agg(lambda lists: sorted(set().union(*lists)))
)

df_clean = (
    df_clean
    .drop(columns='genres')
    .drop_duplicates(song_key)  # already sorted by popularity, so the most popular version is kept
    .join(genres_by_song, on=song_key)
    .sort_values('index')       # restore the original dataset order
    .reset_index(drop=True)
)

print(f"Rows removed (same name + artists): {rows_before - len(df_clean):,}")
print(f"Tracks left: {len(df_clean):,}")
assert not df_clean.duplicated('track_id').any()
assert not df_clean.duplicated(song_key).any()

In [ ]:
# Dedup side effect: the dataset is no longer balanced by genre.
# Genres whose tracks overlap with others (e.g. pop, rock) lose more rows.
genre_counts = df_clean['track_genre'].value_counts()
print(f"Tracks per genre after dedup: min {genre_counts.min()}, max {genre_counts.max()}")
display(genre_counts.tail(10))

In [ ]:
# Drop columns that are not audio features (IDs, text, genre, popularity)
cols_to_remove = ['index', 'track_id', 'artists', 'album_name', 'track_name', 'track_genre', 'genres', 'popularity']
df_features = df_clean.drop(columns=cols_to_remove)

print("\nColumns used for clustering:")
print(df_features.columns.tolist())

print("\nData types after filtering:")
print(df_features.dtypes)

# Standardize the features (essential for K-Means, which is distance-based)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_features)

print("\nScaled dataset shape:", df_scaled.shape)

df_scaled = pd.DataFrame(df_scaled, columns=df_features.columns)

df_scaled.head()

## **About the cleaning steps**

To prepare the data for K-Means, we applied a few transformations so that only relevant and
comparable information reaches the algorithm:

**1. Missing values:**
Only 1 row had missing values, and it was dropped. Since the remaining data was complete, no imputation was needed.

**2. Duplicate tracks:**
Duplicates by `track_id` and by name + artists were removed, as explained above.

**3. Columns that do not fit K-Means:**
We dropped *IDs (index, track_id)*, *text (artists, album_name, track_name)*,
*genre (track_genre, genres)* and *popularity*, which could bias the clusters towards "success"
instead of how the tracks sound.

These columns are not numeric attributes that make sense in a distance calculation, which is what K-Means relies on.

**4. Standardization (StandardScaler):**
StandardScaler transforms every feature to:
* mean ≈ 0
* standard deviation ≈ 1

This matters because features such as loudness, tempo and duration_ms have very different
scales. Without standardization, K-Means would give more weight to the features with larger values (e.g. duration).

## **Exploratory data analysis**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

print("\nData types:")
print(df.dtypes)

# Descriptive statistics of the numeric columns
num_cols = ['popularity', 'duration_ms', 'danceability', 'energy', 'loudness',
            'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

stats = df[num_cols].describe()
pd.set_option('display.float_format', lambda x: '%.2f' % x)
print("\nDescriptive statistics:")
display(stats)

# Popularity distribution
plt.figure(figsize=(8,5))
sns.histplot(df['popularity'], bins=30, kde=True, color='mediumseagreen')
plt.title('Track popularity distribution')
plt.xlabel('Popularity')
plt.ylabel('Count')
plt.show()

# Genre distribution (top 20)
top_genres = df['track_genre'].value_counts().head(20)
plt.figure(figsize=(10,6))
sns.barplot(y=top_genres.index, x=top_genres.values, color='mediumseagreen')
plt.title('Top 20 genres')
plt.xlabel('Number of tracks')
plt.ylabel('Genre')
plt.show()

# Correlation between numeric columns
plt.figure(figsize=(12,10))
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', square=True)
plt.title('Correlation matrix of numeric columns')
plt.show()

## **Training K-Means**

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Standardized features (one row per track)
X = df_scaled

# Train K-Means
n_clusters = 8
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
kmeans.fit(X)

# Cluster assigned to each track
clusters = kmeans.labels_

# PCA (Principal Component Analysis) with 6 components
pca = PCA(n_components=6, random_state=42)
X_pca = pca.fit_transform(X)

df_pca = pd.DataFrame(data=X_pca, columns=[f'PC{i+1}' for i in range(6)])
df_pca['cluster'] = clusters

df_pca.head()

In [ ]:
# Choosing the number of clusters: elbow method + silhouette score

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns

X = df_scaled

# Elbow method: shows from which k adding clusters stops paying off
wcss = []
k_values = range(1, 15)

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42)
    model.fit(X)
    wcss.append(model.inertia_)

plt.figure(figsize=(8,5))
plt.plot(k_values, wcss, marker='o')
plt.title('Elbow method')
plt.xlabel('Number of clusters (k)')
plt.ylabel('WCSS (inertia)')
plt.grid(True)
plt.show()


# Silhouette score: measures how well separated the clusters are
scores = []
k_values_sil = range(2, 13)

# Silhouette is O(n²), so use a 10% sample to keep it fast
X_sample = X.sample(frac=0.10, random_state=42)

for k in k_values_sil:
    model = KMeans(n_clusters=k, random_state=42)
    labels = model.fit_predict(X_sample)
    score = silhouette_score(X_sample, labels)
    scores.append(score)
    print(f'Clusters: {k}, Silhouette Score: {score:.4f}')

plt.figure(figsize=(8,5))
plt.plot(k_values_sil, scores, marker='o')
plt.title('Silhouette score by number of clusters')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Silhouette score')
plt.grid(True)
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Clusters projected on the first two principal components
plt.figure(figsize=(10,6))
sns.scatterplot(
    x='PC1', y='PC2',
    hue='cluster',
    data=df_pca,
    palette='tab10',
    alpha=0.6
)
plt.title('Track clusters (K-Means + PCA)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(title='Cluster')
plt.show()

## **3D PCA**

In [ ]:
from sklearn.decomposition import PCA
import pandas as pd

# PCA with 3 components
pca_3d = PCA(n_components=3, random_state=42)
X_pca_3d = pca_3d.fit_transform(df_scaled)  # df_scaled is the standardized dataset

df_pca_3d = pd.DataFrame(data=X_pca_3d, columns=['PC1', 'PC2', 'PC3'])
df_pca_3d['cluster'] = clusters  # K-Means clusters

df_pca_3d.head()

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# One color per cluster
colors = ['#5DF534', '#425B26', '#151716', '#876E03', '#E8CECF',
          '#023411', '#042E54', '#2572D3', '#FD0D0C', '#169C90']
df_pca_3d['color'] = df_pca_3d['cluster'].map({i: colors[i] for i in range(n_clusters)})

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

ax.scatter3D(df_pca_3d['PC1'], df_pca_3d['PC2'], df_pca_3d['PC3'], c=df_pca_3d['color'], alpha=0.6)

ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')
ax.set_title('Track clusters (K-Means + 3D PCA)')

plt.show()

## **Interactive 3D PCA (Plotly)**

In [ ]:
import plotly.express as px

# Add track info for the hover tooltip
df_pca_3d['track_name'] = df_clean['track_name'].values
df_pca_3d['artists'] = df_clean['artists'].values
df_pca_3d['track_genre'] = df_clean['track_genre'].values

fig = px.scatter_3d(
    df_pca_3d,
    x='PC1',
    y='PC2',
    z='PC3',
    color='cluster',
    opacity=0.7,
    hover_data=['track_name', 'artists', 'track_genre'],
    title='Track clusters (K-Means + interactive 3D PCA)'
)

fig.update_traces(marker=dict(size=4))
fig.update_layout(scene=dict(
    xaxis_title='PC1',
    yaxis_title='PC2',
    zaxis_title='PC3'
))

fig.show()

## **Testing recommendations**

In [ ]:
# Search tracks by artist name
lana_tracks = df_pca_3d[df_pca_3d['artists'].str.contains("Lana", case=False, na=False)]

if len(lana_tracks) == 0:
    print("No matching tracks found in the dataset.")
else:
    print(f"Matching tracks found: {len(lana_tracks)}")
    display(lana_tracks[['track_name','artists','track_genre']].head(10))

In [ ]:
from sklearn.metrics.pairwise import euclidean_distances

# Reference track
ref_track = "Brooklyn Baby"

if ref_track not in df_pca_3d['track_name'].values:
    print(f"Track '{ref_track}' not found in the dataset.")
else:
    # Cluster of the reference track
    ref_cluster = df_pca_3d[df_pca_3d['track_name'] == ref_track]['cluster'].values[0]

    # Coordinates of the reference track in PCA space
    ref_coords = df_pca_3d[df_pca_3d['track_name'] == ref_track][['PC1','PC2','PC3']].values

    # Euclidean distance between the reference track and every track
    df_pca_3d['dist'] = euclidean_distances(df_pca_3d[['PC1','PC2','PC3']], ref_coords).flatten()

    # Closest tracks within the same cluster
    closest = df_pca_3d[(df_pca_3d['cluster'] == ref_cluster) &
                        (df_pca_3d['track_name'] != ref_track)].sort_values('dist')

    print(f"Recommendations for '{ref_track}':")
    display(closest[['track_name','artists','track_genre', 'dist']].head(5))

## **Observations about the clusters**

Looking at the clusters produced by K-Means:

* The algorithm **groups tracks by acoustic, numeric features** such as *danceability*, *energy*, *loudness*, *tempo*, *valence*, *instrumentalness*, etc.
* As a result, tracks in the same cluster share **similar sound profiles**: comparable rhythm, intensity and tonality.
* However, **these similarities do not necessarily reflect genre, theme or vibe**. Two tracks can be close in acoustic terms even if one is light indie pop and the other is country or melodic rock, with completely different lyrics.
* This happens because the recommendation relies **only on objective audio features**, which capture sound patterns but not context, emotion or narrative.

In [ ]:
from sklearn.metrics.pairwise import euclidean_distances

ref_track = 'Brooklyn Baby'
ref_row = df_pca_3d[df_pca_3d['track_name'] == ref_track]

if ref_row.empty:
    print(f"Track '{ref_track}' not found in the dataset.")
else:
    ref_cluster = ref_row['cluster'].values[0]

    # Only tracks from the same cluster
    cluster_songs = df_pca_3d[df_pca_3d['cluster'] == ref_cluster].copy()

    # Weighted Euclidean distance in 3D PCA space
    weights = [1.0, 1.0, 0.8]
    ref_coords = ref_row[['PC1','PC2','PC3']].values
    distances = euclidean_distances(cluster_songs[['PC1','PC2','PC3']] * weights, ref_coords * weights).flatten()
    cluster_songs['dist'] = distances

    # Only tracks from the same genre
    ref_genre = ref_row['track_genre'].values[0]
    cluster_songs = cluster_songs[cluster_songs['track_genre'] == ref_genre]

    # Exclude the reference track itself
    cluster_songs = cluster_songs[cluster_songs['track_name'] != ref_track]

    recommendations = cluster_songs.sort_values('dist').head(5)

    display(recommendations[['track_name', 'artists', 'track_genre', 'dist']])